# When Coarse and Fine Agree (CODA-CS)

**Paper Companion:** Section 2 (Theorem CS-T1) & Section 3 (Theorem CS-T2).

Two central mathematical results govern observations and descent in CODA-CS:
1. **Theorem CS-T1 (Refinement Preserves Descent):** If a query $q$ is already constant on coarse fibers, it is automatically constant on fine fibers.
2. **Theorem CS-T2 (Family Observation Equality is Pointwise):** A family of observations $\mathrm{observe}_i(x)$ indexed by $i \in I$ agrees across two states $x, y$ if and only if they agree pointwise at every index $i \in I$.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == "jupyter_notebooks" else cwd
code_dir = (root / "code").resolve()
assert (code_dir / "coda_cs_core.py").is_file(), "Run from this package root or jupyter_notebooks/."
sys.path[:] = [str(code_dir)] + [entry for entry in sys.path if entry != str(code_dir)]
import coda_cs_core as companion
assert Path(companion.__file__).resolve() == code_dir / "coda_cs_core.py", "An ambient module shadowed the local companion."
import cochain_presentation as cochain
assert Path(cochain.__file__).resolve() == code_dir / "cochain_presentation.py", "An ambient solver shadowed the local solver."

from sympy import Rational
from coda_cs_core import ContinuumSignal, ProfileMap
print("Dependencies loaded.")

Dependencies loaded.


## 1. The Four States and Fiber Partitions

Let the state space be $X = \mathbf{2} \times \mathbf{2} = \{ (\mathsf{false},\mathsf{false}), (\mathsf{false},\mathsf{true}), (\mathsf{true},\mathsf{false}), (\mathsf{true},\mathsf{true}) \}$.

- **Fine look:** $\mathrm{fine}(x) = x$ (the identity). Each fiber is a singleton:
  $$\mathrm{fine}^{-1}(b) = \{ b \} \quad (4 \text{ fibers of size } 1).$$
- **Forgetful map:** $\mathrm{forget}(b) = b[0]$ (first coordinate).
- **Coarse look:** $p(x) = \mathrm{forget}(\mathrm{fine}(x)) = x[0]$. There are two fibers of size 2:
  $$p^{-1}(\mathsf{false}) = \{ (\mathsf{false},\mathsf{false}), (\mathsf{false},\mathsf{true}) \},$$
  $$p^{-1}(\mathsf{true}) = \{ (\mathsf{true},\mathsf{false}), (\mathsf{true},\mathsf{true}) \}.$$

Because each fine fiber is contained inside a coarse fiber ($F_{\mathrm{fine}} \subseteq F_{\mathrm{coarse}}$), any query constant on the larger coarse fibers must be constant on the smaller fine fibers!

In [2]:
pairs = [(False, False), (False, True), (True, False), (True, True)]
fine = lambda x: x
forget = lambda b: b[0]
coarse = lambda x: forget(fine(x))
first = lambda x: x[0]
second = lambda x: x[1]

def fiber_constant(p, q, domain):
    seen = {}
    for x in domain:
        px, qx = p(x), q(x)
        if px in seen and seen[px] != qx:
            return False
        seen[px] = qx
    return True

print("Query 'first':  constant on coarse?", fiber_constant(coarse, first, pairs))
print("Query 'first':  constant on fine?  ", fiber_constant(fine, first, pairs))
print("Query 'second': constant on coarse?", fiber_constant(coarse, second, pairs))
print("Query 'second': constant on fine?  ", fiber_constant(fine, second, pairs))

assert fiber_constant(coarse, first, pairs) is True
assert fiber_constant(fine, first, pairs) is True
assert fiber_constant(coarse, second, pairs) is False
assert fiber_constant(fine, second, pairs) is True

Query 'first':  constant on coarse? True
Query 'first':  constant on fine?   True
Query 'second': constant on coarse? False
Query 'second': constant on fine?   True


## 2. Exhaustive Verification of Theorem CS-T1 across All 16 Boolean Queries

There are exactly $2^4 = 16$ possible queries $q: \mathbf{2}\times\mathbf{2} \to \mathbf{2}$. We test each of them for coarse and fine fiber constancy.

Theorem CS-T1 guarantees: **whenever coarse constancy is True, fine constancy MUST be True.**

In [3]:
print(f"{'Query ID':<10} | {'q values on 4 pairs':<22} | {'Coarse OK?':<12} | {'Fine OK?':<10} | {'CS-T1 holds?'}")
print("-" * 70)

for idx in range(16):
    # Bit pattern for the 4 truth values
    pattern = tuple(bool((idx >> i) & 1) for i in range(4))
    mapping = dict(zip(pairs, pattern))
    q = lambda x, m=mapping: m[x]

    coarse_ok = fiber_constant(coarse, q, pairs)
    fine_ok = fiber_constant(fine, q, pairs)
    implication = (not coarse_ok) or fine_ok

    pat_str = str(tuple(int(v) for v in pattern))
    print(f"Q_{idx:<7} | {pat_str:<22} | {str(coarse_ok):<12} | {str(fine_ok):<10} | {implication}")
    assert implication, f"Theorem CS-T1 violated at query {idx}!"

print("-" * 70)
print("VERIFIED: 16/16 queries satisfy Theorem CS-T1 (Refinement Preserves Descent).")

Query ID   | q values on 4 pairs    | Coarse OK?   | Fine OK?   | CS-T1 holds?
----------------------------------------------------------------------
Q_0       | (0, 0, 0, 0)           | True         | True       | True
Q_1       | (1, 0, 0, 0)           | False        | True       | True
Q_2       | (0, 1, 0, 0)           | False        | True       | True
Q_3       | (1, 1, 0, 0)           | True         | True       | True
Q_4       | (0, 0, 1, 0)           | False        | True       | True
Q_5       | (1, 0, 1, 0)           | False        | True       | True
Q_6       | (0, 1, 1, 0)           | False        | True       | True
Q_7       | (1, 1, 1, 0)           | False        | True       | True
Q_8       | (0, 0, 0, 1)           | False        | True       | True
Q_9       | (1, 0, 0, 1)           | False        | True       | True
Q_10      | (0, 1, 0, 1)           | False        | True       | True
Q_11      | (1, 1, 0, 1)           | False        | True       | True
Q_12      

## 3. Theorem CS-T2: Family Observations Agree Pointwise

Let $\mathrm{observe}: (i: I) \to (X \to P_i)$ be a family of observations. Theorem CS-T2 proves that two states $x, y$ agree under the whole family if and only if they agree at every individual observation:
$$
(i \mapsto \mathrm{observe}_i(x)) = (i \mapsto \mathrm{observe}_i(y))
\iff \forall i.\ \mathrm{observe}_i(x) = \mathrm{observe}_i(y).
$$
Writing a family as a vector creates no 'emergent' equality; agreement is strictly pointwise.

In [4]:
# Case 1: Coordinate pair observation family
observe = (first, second)
family = lambda x: tuple(o(x) for o in observe)

x = (False, True)
y = (False, True)
z = (False, False)

print(f"family(x) = {family(x)}")
print(f"family(y) = {family(y)}")
print(f"family(z) = {family(z)}")

assert (family(x) == family(y)) == all(observe[i](x) == observe[i](y) for i in range(2))
assert (family(x) == family(z)) == all(observe[i](x) == observe[i](z) for i in range(2))

# Case 2: Signal point evaluations on a grid profile
pts = (Rational(0, 1), Rational(1, 2), Rational(1, 1))
prof = ProfileMap(profile_id="grid3", sample_points=pts)
sig_a = ContinuumSignal("a", {t: t for t in pts})
sig_b = ContinuumSignal("b", {t: t for t in pts})
sig_c = ContinuumSignal("c", {t: 2 * t for t in pts})

proj_a = prof.project(sig_a)
proj_b = prof.project(sig_b)
proj_c = prof.project(sig_c)

print("Profiles equal (a vs b)?", proj_a == proj_b)
print("Pointwise equal (a vs b)?", all(sig_a.evaluate_at(t) == sig_b.evaluate_at(t) for t in pts))
print("Profiles equal (a vs c)?", proj_a == proj_c)
print("Pointwise equal (a vs c)?", all(sig_a.evaluate_at(t) == sig_c.evaluate_at(t) for t in pts))

assert (proj_a == proj_b) == all(sig_a.evaluate_at(t) == sig_b.evaluate_at(t) for t in pts)
assert (proj_a == proj_c) == all(sig_a.evaluate_at(t) == sig_c.evaluate_at(t) for t in pts)
print("VERIFIED: Family equality is strictly equivalent to pointwise equality at each index.")

family(x) = (False, True)
family(y) = (False, True)
family(z) = (False, False)
Profiles equal (a vs b)? True
Pointwise equal (a vs b)? True
Profiles equal (a vs c)? False
Pointwise equal (a vs c)? False
VERIFIED: Family equality is strictly equivalent to pointwise equality at each index.
